In [ ]:
import json
import warnings
from pathlib import Path

import pandas as pd

In [ ]:
def process_llm_results(input_path, output_path=None):
    """Expand column matches in raw LLM results into processed-result rows."""
    input_path = Path(input_path)
    if output_path is None:
        output_path = input_path.with_name(
            f"{input_path.stem}_processed{input_path.suffix}"
        )
    output_path = Path(output_path)

    results = pd.read_csv(input_path)
    required_columns = {"LEFT_TABLE", "RIGHT_TABLE", "RESPONSE"}
    missing_columns = required_columns.difference(results.columns)
    if missing_columns:
        raise ValueError(f"Input CSV is missing required columns: {sorted(missing_columns)}")

    processed_rows = []
    for row_number, row in enumerate(results.itertuples(index=False), start=2):
        row = row._asdict()
        try:
            response = json.loads(row["RESPONSE"])
        except (json.JSONDecodeError, TypeError) as error:
            raise ValueError(f"Invalid RESPONSE JSON on CSV row {row_number}") from error

        if not isinstance(response, dict):
            raise ValueError(f"RESPONSE on CSV row {row_number} must be a JSON object")
        if str(response.get("table_joinable", "")).strip().casefold() != "yes":
            continue
        if str(response.get("column_joinable", "")).strip().casefold() != "yes":
            continue

        target_match = response.get("target_match")
        candidate_match = response.get("candidate_match")
        if target_match is None and candidate_match is None:
            warnings.warn(
                f"Skipping joinable response with null matches on CSV row {row_number}",
                stacklevel=2,
            )
            continue
        if isinstance(target_match, str):
            target_match = [target_match]
        if isinstance(candidate_match, str):
            candidate_match = [candidate_match]
        if not isinstance(target_match, list) or not isinstance(candidate_match, list):
            raise ValueError(
                f"Joinable RESPONSE on CSV row {row_number} must contain target_match "
                "and candidate_match lists"
            )
        if len(target_match) != len(candidate_match):
            warnings.warn(
                f"Skipping mismatched column-match lists on CSV row {row_number} "
                f"({len(target_match)} target, {len(candidate_match)} candidate)",
                stacklevel=2,
            )
            continue

        left_table = row["LEFT_TABLE"]
        right_table = row["RIGHT_TABLE"]
        for left_column, right_column in zip(target_match, candidate_match):
            if not isinstance(left_column, str) or not isinstance(right_column, str):
                raise ValueError(f"Column names must be strings on CSV row {row_number}")

            if left_table > right_table:
                table_1, table_2 = right_table, left_table
                column_1, column_2 = right_column, left_column
            else:
                table_1, table_2 = left_table, right_table
                column_1, column_2 = left_column, right_column

            processed_rows.append(
                {
                    "ds_name": table_1,
                    "ds_name_2": table_2,
                    "left_column": column_1,
                    "right_column": column_2,
                }
            )

    columns = ["ds_name", "ds_name_2", "left_column", "right_column"]
    processed = pd.DataFrame(processed_rows, columns=columns)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    processed = processed.drop_duplicates()
    processed.to_csv(output_path, index=False)
    return processed

In [ ]:
project_dir = Path.cwd()
BENCHMARK = 'OpenData' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider
MODEL = "gpt-5.6-luna"
INPUT_PATH = project_dir / BENCHMARK / "output" / f"{MODEL}_results.csv"
OUTPUT_PATH = project_dir / BENCHMARK / "output" / f"{MODEL}_results_processed.csv"

processed_results = process_llm_results(
    INPUT_PATH,
    OUTPUT_PATH,
)
processed_results.head()